# 2.2 Data Description

> **CRISP-DM Phase:** 2. Data Understanding | **Task:** 2.2 Describe Data
>
> This notebook profiles all acquired datasets to produce a comprehensive data dictionary, surface statistics, and initial observations.

**Datasets:**
1. `train.csv` — 891 passengers with survival labels
2. `test.csv` — 418 passengers to predict
3. `gender_submission.csv` — Baseline submission (females survive)

**Source Documents:**
- 2.1 Data Collection Report: `docs/crisp-dm/2-data-understanding/2.1-data-collection.md`
- 1.3 Data Mining Goals: `docs/crisp-dm/1-business-understanding/1.3-data-mining-goals.md`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# --- Project root resolution (works in VS Code and terminal) ---
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if (PROJECT_ROOT / "notebooks").is_dir():
    pass  # cwd is project root
elif (PROJECT_ROOT.parent / "notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent  # cwd is a subdirectory

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "titanic"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Files: {[f.name for f in DATA_DIR.glob('*.csv')]}")

Project root: /Users/tba8ydd/Documents/claude-template
Data directory: /Users/tba8ydd/Documents/claude-template/data/raw/titanic
Files: ['test.csv', 'train.csv', 'gender_submission.csv']


In [2]:
# Load all datasets
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
baseline = pd.read_csv(DATA_DIR / "gender_submission.csv")

datasets = {"train.csv": train, "test.csv": test, "gender_submission.csv": baseline}

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  Duplicated rows: {df.duplicated().sum()}")


  train.csv
  Shape: 891 rows x 12 columns
  Memory: 285.6 KB
  Duplicated rows: 0

  test.csv
  Shape: 418 rows x 11 columns
  Memory: 131.0 KB
  Duplicated rows: 0

  gender_submission.csv
  Shape: 418 rows x 2 columns
  Memory: 6.7 KB
  Duplicated rows: 0


## Dataset 1: train.csv — Column-Level Profiling

**Grain:** One row per passenger (unique PassengerId)
**Target variable:** `Survived` (0 = deceased, 1 = survived)

In [3]:
# Data types and basic info
print("--- Data Types ---")
print(train.dtypes)
print(f"\n--- Shape: {train.shape} ---")

--- Data Types ---
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

--- Shape: (891, 12) ---


In [4]:
# Null counts and percentages for train.csv
null_info = pd.DataFrame({
    "dtype": train.dtypes,
    "non_null": train.notnull().sum(),
    "null_count": train.isnull().sum(),
    "null_pct": (train.isnull().mean() * 100).round(1),
    "unique": train.nunique(),
})
null_info

,dtype,non_null,null_count,null_pct,unique
PassengerId,int64,891,0,0.0,891
Survived,int64,891,0,0.0,2
Pclass,int64,891,0,0.0,3
Name,str,891,0,0.0,891
Sex,str,891,0,0.0,2
Age,float64,714,177,19.9,88
SibSp,int64,891,0,0.0,7
Parch,int64,891,0,0.0,7
Ticket,str,891,0,0.0,681
Fare,float64,891,0,0.0,248


In [5]:
# Numeric field statistics — train.csv
train.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
# Categorical field statistics — train.csv
cat_cols = train.select_dtypes(exclude="number").columns.tolist()
print(f"Categorical columns: {cat_cols}")
train[cat_cols].describe()

Categorical columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


In [7]:
# Per-column detailed profiling — train.csv
for col in train.columns:
    print(f"\n{'─'*50}")
    print(f"  {col}  (dtype: {train[col].dtype})")
    print(f"{'─'*50}")
    print(f"  Nulls: {train[col].isnull().sum()} ({train[col].isnull().mean()*100:.1f}%)")
    print(f"  Unique: {train[col].nunique()}")

    if pd.api.types.is_numeric_dtype(train[col]):
        print(f"  Min: {train[col].min()}, Max: {train[col].max()}")
        print(f"  Mean: {train[col].mean():.2f}, Std: {train[col].std():.2f}")
        print(f"  Median: {train[col].median():.2f}")
    else:
        vc = train[col].value_counts().head(5)
        print(f"  Top values:")
        for val, cnt in vc.items():
            print(f"    {val}: {cnt} ({cnt/len(train)*100:.1f}%)")


──────────────────────────────────────────────────
  PassengerId  (dtype: int64)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 891
  Min: 1, Max: 891
  Mean: 446.00, Std: 257.35
  Median: 446.00

──────────────────────────────────────────────────
  Survived  (dtype: int64)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 2
  Min: 0, Max: 1
  Mean: 0.38, Std: 0.49
  Median: 0.00

──────────────────────────────────────────────────
  Pclass  (dtype: int64)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 3
  Min: 1, Max: 3
  Mean: 2.31, Std: 0.84
  Median: 3.00

──────────────────────────────────────────────────
  Name  (dtype: str)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 891
  Top values:
    Braund, Mr. Owen Harris: 1 (0.1%)
    Cumings, Mrs. John Bradley (Florence Briggs Thayer): 1 (0.1%)
    Heikkinen, Miss. Laina: 1 (0.1%)
    Futrelle, Mrs. Jacques Hea

In [8]:
# Target variable distribution
print("--- Survived Distribution (train.csv) ---")
surv = train["Survived"].value_counts()
for val, cnt in surv.items():
    label = "Survived" if val == 1 else "Deceased"
    print(f"  {val} ({label}): {cnt} ({cnt/len(train)*100:.1f}%)")
print(f"\n  Class imbalance ratio: {surv[0]/surv[1]:.1f}:1")

--- Survived Distribution (train.csv) ---
  0 (Deceased): 549 (61.6%)
  1 (Survived): 342 (38.4%)

  Class imbalance ratio: 1.6:1


In [9]:
# Pclass distribution
print("--- Pclass Distribution (train.csv) ---")
pclass = train["Pclass"].value_counts().sort_index()
for val, cnt in pclass.items():
    labels = {1: "Upper", 2: "Middle", 3: "Lower"}
    print(f"  {val} ({labels[val]}): {cnt} ({cnt/len(train)*100:.1f}%)")

--- Pclass Distribution (train.csv) ---
  1 (Upper): 216 (24.2%)
  2 (Middle): 184 (20.7%)
  3 (Lower): 491 (55.1%)


In [10]:
# Cabin deck extraction — train.csv
cabin_decks = train["Cabin"].dropna().str[0].value_counts().sort_index()
print("--- Cabin Deck Distribution (train.csv, non-null only) ---")
for deck, cnt in cabin_decks.items():
    print(f"  Deck {deck}: {cnt}")
print(f"\n  Total with cabin info: {train['Cabin'].notna().sum()} / {len(train)} ({train['Cabin'].notna().mean()*100:.1f}%)")

--- Cabin Deck Distribution (train.csv, non-null only) ---
  Deck A: 15
  Deck B: 47
  Deck C: 59
  Deck D: 33
  Deck E: 32
  Deck F: 13
  Deck G: 4
  Deck T: 1

  Total with cabin info: 204 / 891 (22.9%)


In [11]:
# Ticket analysis — shared tickets
ticket_counts = train["Ticket"].value_counts()
shared_tickets = ticket_counts[ticket_counts > 1]
print(f"--- Ticket Sharing (train.csv) ---")
print(f"  Total unique tickets: {train['Ticket'].nunique()}")
print(f"  Shared tickets (>1 passenger): {len(shared_tickets)}")
print(f"  Passengers on shared tickets: {shared_tickets.sum()}")
print(f"\n  Top 5 most-shared tickets:")
for ticket, cnt in shared_tickets.head(5).items():
    print(f"    {ticket}: {cnt} passengers")

# Ticket prefix vs numeric-only
has_prefix = train["Ticket"].str.contains(r"[A-Za-z]", na=False)
print(f"\n  Tickets with alpha prefix: {has_prefix.sum()}")
print(f"  Numeric-only tickets: {(~has_prefix).sum()}")

--- Ticket Sharing (train.csv) ---
  Total unique tickets: 681
  Shared tickets (>1 passenger): 134
  Passengers on shared tickets: 344

  Top 5 most-shared tickets:
    347082: 7 passengers
    1601: 7 passengers
    CA. 2343: 7 passengers
    3101295: 6 passengers
    CA 2144: 6 passengers

  Tickets with alpha prefix: 230
  Numeric-only tickets: 661


In [12]:
# Zero-fare investigation
zero_fare = train[train["Fare"] == 0]
print(f"--- Zero-Fare Passengers (train.csv) ---")
print(f"  Count: {len(zero_fare)}")
print(f"\n  Pclass distribution:")
print(zero_fare["Pclass"].value_counts().to_string())
print(f"\n  Embarked distribution:")
print(zero_fare["Embarked"].value_counts().to_string())

--- Zero-Fare Passengers (train.csv) ---
  Count: 15

  Pclass distribution:
Pclass
2    6
1    5
3    4

  Embarked distribution:
Embarked
S    15


## Dataset 2: test.csv — Column-Level Profiling

**Grain:** One row per passenger (unique PassengerId, range 892–1309)
**Note:** No `Survived` column — this is the prediction target.

In [13]:
# Null counts and percentages for test.csv
null_info_test = pd.DataFrame({
    "dtype": test.dtypes,
    "non_null": test.notnull().sum(),
    "null_count": test.isnull().sum(),
    "null_pct": (test.isnull().mean() * 100).round(1),
    "unique": test.nunique(),
})
null_info_test

,dtype,non_null,null_count,null_pct,unique
PassengerId,int64,418,0,0.0,418
Pclass,int64,418,0,0.0,3
Name,str,418,0,0.0,418
Sex,str,418,0,0.0,2
Age,float64,332,86,20.6,79
SibSp,int64,418,0,0.0,7
Parch,int64,418,0,0.0,8
Ticket,str,418,0,0.0,363
Fare,float64,417,1,0.2,169
Cabin,str,91,327,78.2,76


In [14]:
# Numeric field statistics — test.csv
test.describe()

,PassengerId,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [15]:
# Categorical field statistics — test.csv
cat_cols_test = test.select_dtypes(exclude="number").columns.tolist()
print(f"Categorical columns: {cat_cols_test}")
test[cat_cols_test].describe()

Categorical columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


,Name,Sex,Ticket,Cabin,Embarked
count,418,418,418,91,418
unique,418,2,363,76,3
top,"Kelly, Mr. James",male,PC 17608,B57 B59 B63 B66,S
freq,1,266,5,3,270


In [16]:
# Per-column detailed profiling — test.csv
for col in test.columns:
    print(f"\n{'─'*50}")
    print(f"  {col}  (dtype: {test[col].dtype})")
    print(f"{'─'*50}")
    print(f"  Nulls: {test[col].isnull().sum()} ({test[col].isnull().mean()*100:.1f}%)")
    print(f"  Unique: {test[col].nunique()}")

    if pd.api.types.is_numeric_dtype(test[col]):
        print(f"  Min: {test[col].min()}, Max: {test[col].max()}")
        print(f"  Mean: {test[col].mean():.2f}, Std: {test[col].std():.2f}")
        print(f"  Median: {test[col].median():.2f}")
    else:
        vc = test[col].value_counts().head(5)
        print(f"  Top values:")
        for val, cnt in vc.items():
            print(f"    {val}: {cnt} ({cnt/len(test)*100:.1f}%)")


──────────────────────────────────────────────────
  PassengerId  (dtype: int64)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 418
  Min: 892, Max: 1309
  Mean: 1100.50, Std: 120.81
  Median: 1100.50

──────────────────────────────────────────────────
  Pclass  (dtype: int64)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 3
  Min: 1, Max: 3
  Mean: 2.27, Std: 0.84
  Median: 3.00

──────────────────────────────────────────────────
  Name  (dtype: str)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 418
  Top values:
    Kelly, Mr. James: 1 (0.2%)
    Wilkes, Mrs. James (Ellen Needs): 1 (0.2%)
    Myles, Mr. Thomas Francis: 1 (0.2%)
    Wirz, Mr. Albert: 1 (0.2%)
    Hirvonen, Mrs. Alexander (Helga E Lindqvist): 1 (0.2%)

──────────────────────────────────────────────────
  Sex  (dtype: str)
──────────────────────────────────────────────────
  Nulls: 0 (0.0%)
  Unique: 2
  Top values:
    mal

## Dataset 3: gender_submission.csv — Baseline Reference

**Grain:** One row per test passenger (PassengerId 892–1309)
**Purpose:** Submission format template; gender-only baseline (~76.5% accuracy)

In [17]:
# gender_submission.csv profiling
print(f"Shape: {baseline.shape}")
print(f"Columns: {list(baseline.columns)}")
print(f"PassengerId range: {baseline['PassengerId'].min()} – {baseline['PassengerId'].max()}")
print(f"\nSurvived distribution:")
surv_base = baseline["Survived"].value_counts()
for val, cnt in surv_base.items():
    print(f"  {val}: {cnt} ({cnt/len(baseline)*100:.1f}%)")
print(f"\nAll PassengerIds match test.csv: {set(baseline['PassengerId']) == set(test['PassengerId'])}")

Shape: (418, 2)
Columns: ['PassengerId', 'Survived']
PassengerId range: 892 – 1309

Survived distribution:
  0: 266 (63.6%)
  1: 152 (36.4%)

All PassengerIds match test.csv: True


## Train/Test Distribution Comparison

Comparing feature distributions between train and test sets to check for covariate shift.

In [18]:
# Compare numeric feature distributions between train and test
shared_numeric = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

print("--- Train vs Test: Numeric Features (mean / std) ---")
print(f"{'Field':<15} {'Train Mean':>12} {'Test Mean':>12} {'Train Std':>12} {'Test Std':>12}")
print("─" * 65)
for col in shared_numeric:
    print(f"{col:<15} {train[col].mean():>12.2f} {test[col].mean():>12.2f} "
          f"{train[col].std():>12.2f} {test[col].std():>12.2f}")

# Compare categorical distributions
print("\n--- Train vs Test: Categorical Features (%) ---")
for col in ["Sex", "Embarked"]:
    print(f"\n  {col}:")
    train_pct = train[col].value_counts(normalize=True) * 100
    test_pct = test[col].value_counts(normalize=True) * 100
    for val in train_pct.index:
        t_pct = train_pct.get(val, 0)
        te_pct = test_pct.get(val, 0)
        print(f"    {val}: train {t_pct:.1f}% / test {te_pct:.1f}%")

# Compare null rates
print("\n--- Train vs Test: Null Rates (%) ---")
shared_cols = [c for c in train.columns if c in test.columns and c != "Survived"]
for col in shared_cols:
    tn = train[col].isnull().mean() * 100
    ten = test[col].isnull().mean() * 100
    if tn > 0 or ten > 0:
        print(f"  {col}: train {tn:.1f}% / test {ten:.1f}%")

--- Train vs Test: Numeric Features (mean / std) ---
Field             Train Mean    Test Mean    Train Std     Test Std
─────────────────────────────────────────────────────────────────
Pclass                  2.31         2.27         0.84         0.84
Age                    29.70        30.27        14.53        14.18
SibSp                   0.52         0.45         1.10         0.90
Parch                   0.38         0.39         0.81         0.98
Fare                   32.20        35.63        49.69        55.91

--- Train vs Test: Categorical Features (%) ---

  Sex:
    male: train 64.8% / test 63.6%
    female: train 35.2% / test 36.4%

  Embarked:
    S: train 72.4% / test 64.6%
    C: train 18.9% / test 24.4%
    Q: train 8.7% / test 11.0%

--- Train vs Test: Null Rates (%) ---
  Age: train 19.9% / test 20.6%
  Fare: train 0.0% / test 0.2%
  Cabin: train 77.1% / test 78.2%
  Embarked: train 0.2% / test 0.0%


## Structural Notes

### Join Keys
| Dataset A | Key Field(s) | Dataset B | Key Field(s) | Relationship |
|-----------|-------------|-----------|-------------|--------------|
| train.csv | PassengerId | — | — | Self-contained |
| test.csv | PassengerId | gender_submission.csv | PassengerId | 1:1 |

PassengerId ranges are disjoint: train 1–891, test 892–1309.

### Format Details
| Dataset | Format | Encoding | Delimiter | Header Row |
|---------|--------|----------|-----------|------------|
| train.csv | CSV | UTF-8 | Comma | Yes |
| test.csv | CSV | UTF-8 | Comma | Yes |
| gender_submission.csv | CSV | UTF-8 | Comma | Yes |

## Initial Observations

### Red Flags
| # | Dataset | Field(s) | Observation | Severity | Action Needed |
|---|---------|----------|-------------|----------|---------------|
| 1 | train + test | Cabin | 77–78% missing — too sparse for direct use | High | Extract deck letter; create `has_cabin` flag; check if missingness correlates with class/survival |
| 2 | train + test | Age | ~20% missing — key predictor | Medium | Impute using title-group medians; validate via CV |
| 3 | train | Fare | 15 passengers with Fare=0 | Low | Investigate: crew, children, or data errors? |
| 4 | test | Fare | 1 missing value (PassengerId 1044) | Low | Impute with median fare for same Pclass |
| 5 | train | Embarked | 2 missing values | Low | Impute with mode ('S') |
| 6 | train | Cabin | Deck 'T' appears once — likely anomaly | Low | Investigate; may merge with nearest deck |

### Noteworthy Patterns
- **Class imbalance is moderate** (38.4% survived) — stratified CV essential, no resampling needed
- **Pclass skewed toward 3rd class** (55.1%)
- **Male majority** (~64% in both sets) — consistent across train/test
- **Fare heavily right-skewed** (mean 32.20, median 14.45, max 512.33) — log transform likely helpful
- **SibSp and Parch zero-inflated** (median=0) — most passengers traveled alone
- **Ticket sharing:** 681 unique tickets for 891 passengers — family/group travel signal
- **Multi-cabin entries** in Cabin (e.g., "B57 B59 B63 B66")
- **Train/test distributions are comparable** — no obvious covariate shift

### Contradictions with Prior Documents
- None found. All observations align with 1.2 and 1.3 expectations.